# Paper Plots — Modularized Workflow

This notebook reproduces the dN/dX, CDDF f(N,z), and Omega_HI plots for the DESI GP-DLA paper.
It uses the extracted Python modules rather than inline notebook code:

- `CDDF_analysis/cddf_mock.py` — search windows, dN/dX, CDDF, Omega_HI
- `CDDF_analysis/cddf_calibration.py` — calibration factors (alpha, correction ratio)
- `CDDF_analysis/cddf_io.py` — save/load calibrated text tables

**See also:** `docs/tutorial_population_statistics.md` for a walkthrough of the workflow.

---

## Workflow overview

```
London mock catalogs (dla_cat.fits, zcat.fits)
  → compute_dndx() on mock [measured]
  → compute_dndx() on truth catalog
  → calibration_factor_alpha() → alpha(z)
  → apply_alpha_to_bounds() on real DESI data → calibrated dN/dX
  → save_dndx_combined() → text table

London mock catalogs
  → compute_cddf_fN() on mock [measured]
  → truth_cddf_prochaska2014() → truth f(N,z)
  → correction_ratio_with_uncertainty() → r(logN, z)
  → apply_correction_with_uncertainty() on real DESI data → calibrated CDDF
  → save_all_cddf_txt_tables() → per-z-bin text tables
```

## 0. Imports and paths

In [ ]:
import sys
import os

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

# Add CDDF_analysis to path
sys.path.insert(0, "../CDDF_analysis")
import cddf_mock as cm
import cddf_calibration as cal
import cddf_io as cio

plt.rcParams.update({'font.size': 12})

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
# London mock catalogs (NERSC path or local copy)
MOCK_DLA_CAT  = "../data/london/dla_cat.fits"        # GP-DLA output on mock spectra
MOCK_TRUTH_CAT = "../data/london/dla_truth_cat.fits" # Truth absorber catalog from mock
MOCK_QSO_CAT  = "../data/london/zcat.fits"           # Mock QSO catalog

# Real DESI LOA catalogs (NERSC path)
REAL_DLA_CAT  = "../data/loa/dlacat-loa-main-dark.fits"
REAL_QSO_CAT  = "../data/loa/QSO_cat_loa_main_dark_healpix_v2.fits"

# Output directory for text tables
OUTDIR = "../output/paper_tables"
os.makedirs(OUTDIR, exist_ok=True)

# ── Global search window parameters (must match inference settings) ─────────
# See constants.py and docs/tutorial_population_statistics.md
ZMIN         = 2.15       # floor on absorber redshift
V_PROX_KMS   = 3000.0    # proximity zone velocity cut [km/s]
LAMBDA_OBS_MIN = 3700.0  # DESI blue cutoff [Å]

# SNR cuts (match notebook analysis)
SNR_CUT_DLA    = 4.0
SNR_CUT_SUBDLA = 6.0
SNR_CUT_LLS    = 6.0

## 1. Load catalogs

In [ ]:
mock_dla  = Table.read(MOCK_DLA_CAT)
mock_qso  = Table.read(MOCK_QSO_CAT)
truth_cat = Table.read(MOCK_TRUTH_CAT)

real_dla  = Table.read(REAL_DLA_CAT)
real_qso  = Table.read(REAL_QSO_CAT)

print(f"Mock DLA catalog:    {len(mock_dla):,} absorbers")
print(f"Mock QSO catalog:    {len(mock_qso):,} QSOs")
print(f"Truth catalog:       {len(truth_cat):,} absorbers")
print(f"Real DLA catalog:    {len(real_dla):,} absorbers")
print(f"Real QSO catalog:    {len(real_qso):,} QSOs")

## 2. Redshift bins and search-window parameters

In [ ]:
# DLA redshift bins (z_mid → edges via zbins_from_zmid_uniform)
z_mid_dla = np.array([2.3, 2.7, 3.1, 3.5])
zbins_dla = cm.zbins_from_zmid_uniform(z_mid_dla)

# LLS/subDLA redshift bins
z_mid_lls = np.array([2.15, 2.5, 2.85, 3.2, 3.55])
zbins_lls = cm.zbins_from_zmid_uniform(z_mid_lls)

# Shared keyword arguments for dN/dX computation
COMMON_KW = dict(
    v_prox_kms=V_PROX_KMS,
    zmin=ZMIN,
    lambda_obs_min=LAMBDA_OBS_MIN,
    blue_limit_mode="max",
    n_boot=200,
)

# NHI range definitions
NHI_DLA    = (20.3, 23.0)
NHI_SUBDLA = (19.0, 20.3)
NHI_LLS    = (17.2, 19.0)

## 3. Compute dN/dX — mock measured, mock truth, and real DESI

In [ ]:
# ── DLA dN/dX ──────────────────────────────────────────────────────────────
# Apply SNR cut before passing catalogs
mock_dla_snr  = mock_dla[mock_dla["snr"] > SNR_CUT_DLA]   # adjust column name
real_dla_snr  = real_dla[real_dla["snr"] > SNR_CUT_DLA]

# Compute dN/dX on mock (measured by GP-DLA)
out_mock_dla = cm.compute_dndx(
    mock_dla_snr, mock_qso, zbins=zbins_dla,
    logNHImin=NHI_DLA[0], logNHImax=NHI_DLA[1],
    **COMMON_KW,
)

# Compute dN/dX on truth catalog
out_truth_dla = cm.compute_dndx(
    truth_cat, mock_qso, zbins=zbins_dla,
    logNHImin=NHI_DLA[0], logNHImax=NHI_DLA[1],
    **COMMON_KW,
)

# Compute dN/dX on real DESI
out_real_dla = cm.compute_dndx(
    real_dla_snr, real_qso, zbins=zbins_dla,
    logNHImin=NHI_DLA[0], logNHImax=NHI_DLA[1],
    **COMMON_KW,
)

## 4. Compute calibration factor alpha(z) and apply to real data

In [ ]:
# Calibration factor alpha(z) = dNdX_truth / dNdX_mock_measured
bounds68_mock = np.column_stack([
    out_mock_dla["boot68_low"],
    out_mock_dla["boot68_high"],
])
cal_dla = cal.calibration_factor_alpha(
    z_meas=out_mock_dla["z_mid"],
    y_meas=out_mock_dla["dndx"],
    bounds68_meas=bounds68_mock,
    out_truth=out_truth_dla,
    truth_y_key="dndx",
)

print("alpha(z) for DLAs:", cal_dla["alpha"])
print("alpha_err:        ", cal_dla["alpha_err"])

In [ ]:
# Apply alpha to real DESI dN/dX
corr_dla = cal.apply_alpha_to_dndx_bounds(
    z_cent=out_real_dla["z_mid"],
    y=out_real_dla["dndx"],
    y68_low=out_real_dla["boot68_low"],
    y68_high=out_real_dla["boot68_high"],
    y95_low=out_real_dla["boot95_low"],
    y95_high=out_real_dla["boot95_high"],
    cal=cal_dla,
)

# Save to text table
cio.save_dndx_combined(
    os.path.join(OUTDIR, "dndx_dla_calibrated.txt"),
    z=corr_dla["z_cent"],
    y_raw=corr_dla["y_raw"],
    y68_raw=np.column_stack([out_real_dla["boot68_low"], out_real_dla["boot68_high"]]),
    y95_raw=np.column_stack([out_real_dla["boot95_low"], out_real_dla["boot95_high"]]),
    y_calibrated=corr_dla["y_corr"],
    y68_calibrated=np.column_stack([corr_dla["y68_low_corr"], corr_dla["y68_high_corr"]]),
    y95_calibrated=np.column_stack([corr_dla["y95_low_corr"], corr_dla["y95_high_corr"]]),
    meta={"absorber": "DLA", "logNHI_min": NHI_DLA[0], "logNHI_max": NHI_DLA[1]},
)

## 5. Compute CDDF f(N,z)

In [ ]:
# log NHI bins (DLA range)
logN_mids_dla = np.array([20.4, 20.6, 20.8, 21.0, 21.2, 21.5, 22.0])
logN_bins_dla = cm.logN_bins_from_mids(logN_mids_dla)

# Compute f(N,z) on mock (measured) and real DESI
out_mock_cddf = cm.compute_cddf_fN(
    mock_dla_snr, mock_qso,
    zbins=zbins_dla,
    logN_bins=logN_bins_dla,
    logNHImin=NHI_DLA[0], logNHImax=NHI_DLA[1],
    **COMMON_KW,
)

out_real_cddf = cm.compute_cddf_fN(
    real_dla_snr, real_qso,
    zbins=zbins_dla,
    logN_bins=logN_bins_dla,
    logNHImin=NHI_DLA[0], logNHImax=NHI_DLA[1],
    **COMMON_KW,
)

## 6. Compute correction ratio r(logN, z) and apply to real data

In [ ]:
# Prochaska+2014 truth spline evaluated at logN bin centers
f_true = 10 ** cm.truth_cddf_prochaska2014(logN_mids_dla)
sig_true = 0.1 * f_true  # approximate; replace with actual truth error if available

# panel_data: list of per-z-bin dicts for saving
panel_data_dla = []

for iz, z_lo in enumerate(zbins_dla[:-1]):
    z_hi = zbins_dla[iz + 1]
    z_mid = 0.5 * (z_lo + z_hi)

    # f(N) from mock (measured by GP-DLA)
    f_mock = out_mock_cddf["fN"][iz]          # adjust key name from actual output
    sig_mock = out_mock_cddf["fN_err"][iz]

    # f(N) from real DESI
    f_real = out_real_cddf["fN"][iz]
    sig_real = out_real_cddf["fN_err"][iz]

    # Correction ratio
    r, sig_r = cal.correction_ratio_with_uncertainty(f_true, sig_true, f_mock, sig_mock)

    # Apply to real data
    f_corr, sig_corr, f68_corr = cal.apply_correction_with_uncertainty(f_real, sig_real, r, sig_r)
    _, _, f95_corr = cal.apply_correction_with_uncertainty(f_real, sig_real, r, sig_r, nsig=2.0)

    panel_data_dla.append(dict(
        title=rf"${z_lo:.1f} \le z < {z_hi:.1f}$",
        logN=logN_mids_dla,
        f_corr=f_corr,
        f68_corr=f68_corr,
        f95_corr=f95_corr,
        f_raw=f_real,
        f68_raw=np.column_stack([f_real - sig_real, f_real + sig_real]),
        f95_raw=np.column_stack([f_real - 2*sig_real, f_real + 2*sig_real]),
        r=r, r68=np.column_stack([r - sig_r, r + sig_r]),
        r95=np.column_stack([r - 2*sig_r, r + 2*sig_r]),
        f_true=f_true, sig_true=sig_true,
    ))

# Save all z-bin CDDF tables
cio.save_all_cddf_txt_tables(
    panel_data_dla,
    outdir=os.path.join(OUTDIR, "cddf_dla"),
    prefix="cddf_dla_calibrated",
)

## 7. dN/dX → dN/dz conversion (ell(z))

In [ ]:
# Convert calibrated dN/dX to dN/dz
z_dla = corr_dla["z_cent"]
ellz_cal = cm.dndx_to_ellz(z_dla, corr_dla["y_corr"])
ellz_bounds68 = cm.dndx_bounds_to_ellz(
    z_dla,
    np.column_stack([corr_dla["y68_low_corr"], corr_dla["y68_high_corr"]])
)

print("dN/dz (calibrated):", ellz_cal)
print("68% bounds:")
print(ellz_bounds68)

## 8. Omega_HI

In [ ]:
# Omega_HI from real DESI calibrated CDDF
# (Uses cddf_mock.omega_hi_from_cddf on real data CDDF)
out_omega_real = cm.omega_hi_from_cddf(
    out_real_cddf,
    logNHImin=NHI_DLA[0],
    logNHImax=NHI_DLA[1],
)
print("Omega_HI (raw real DESI):", out_omega_real["omega_hi"])

## 9. Plot: dN/dX vs redshift

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

# Raw real DESI
ax.errorbar(
    out_real_dla["z_mid"],
    out_real_dla["dndx"],
    yerr=[
        out_real_dla["dndx"] - out_real_dla["boot68_low"],
        out_real_dla["boot68_high"] - out_real_dla["dndx"],
    ],
    fmt='o', label='DESI Y3 (raw)', color='C0', alpha=0.5,
)

# Calibrated real DESI
ax.errorbar(
    corr_dla["z_cent"],
    corr_dla["y_corr"],
    yerr=[
        corr_dla["y_corr"] - corr_dla["y68_low_corr"],
        corr_dla["y68_high_corr"] - corr_dla["y_corr"],
    ],
    fmt='s', label='DESI Y3 (calibrated)', color='C0',
)

# Truth from mock
ax.plot(out_truth_dla["z_mid"], out_truth_dla["dndx"],
        'k--', label='Mock truth', alpha=0.7)

ax.set_xlabel('Redshift')
ax.set_ylabel(r'$dN/dX$')
ax.set_title('DLA line density')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "dndx_dla.pdf"), dpi=150)
plt.show()

## 10. Reload saved tables (verification)

In [ ]:
# Load a saved dN/dX table and verify round-trip
import re as _re
saved_path = os.path.join(OUTDIR, "dndx_dla_calibrated.txt")
if os.path.exists(saved_path):
    data = np.loadtxt(saved_path)
    print("Loaded table shape:", data.shape)
    print("Columns: z, dNdX_cal, 68lo_cal, 68hi_cal, 95lo_cal, 95hi_cal, dNdX_raw, ...")

# Load a per-z-bin CDDF table
cddf_dir = os.path.join(OUTDIR, "cddf_dla")
if os.path.isdir(cddf_dir):
    example = sorted(os.listdir(cddf_dir))[0]
    d = cio.load_cddf_txt_table(os.path.join(cddf_dir, example))
    print(f"Loaded CDDF table: {d['title']}, columns: {[k for k in d if k not in ('title','path')]}")